# YouTube Analytics — Data Cleaning & Feature Engineering

**Tujuan Notebook:**
1. Pembersihan data mentah dari YouTube Studio
2. Engineering fitur-fitur penting untuk ML
3. Klasifikasi performa video (performance class)

---

## 0. Setup & Import

In [5]:
import pandas as pd
import numpy as np
import re
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ Library siap!")

✅ Library siap!


## 1. Load Data

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd

FILE_PATH = '/content/drive/MyDrive/Capstune Project tempa/NOTEBOOKS/data/data_fix_dengan_jam.csv'

df_raw = pd.read_csv(FILE_PATH)

print(df_raw.shape)
df_raw.head(3)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Capstune Project tempa/NOTEBOOKS/data/data_fix_dengan_jam.csv'

---
## 2. Pembersihan Data (Data Cleaning)

### 2.1 Hapus baris Total & baris kosong

In [ ]:
# Baris pertama adalah baris 'Total' dari YouTube Studio — hapus
df = df_raw[
    df_raw['Konten'].notna() &
    (df_raw['Konten'] != 'Total')
].copy()

df = df.reset_index(drop=True)
print(f" Setelah hapus baris Total: {len(df)} video")

### 2.2 Hapus kolom yang hampir semua null (>80%)

In [ ]:
THRESHOLD_NULL = 0.80  # Kolom dengan >80% null akan di-drop

null_pct = df.isnull().mean()
cols_to_drop = null_pct[null_pct > THRESHOLD_NULL].index.tolist()

print(f"Kolom yang di-drop ({len(cols_to_drop)} kolom):")
for col in cols_to_drop:
    print(f"  - {col}: {null_pct[col]*100:.1f}% null")

df = df.drop(columns=cols_to_drop)
print(f"\n Shape setelah drop kolom null berat: {df.shape}")

### 2.3 Konversi tipe data

In [ ]:
# ── Konversi waktu tonton format MM:SS atau H:MM:SS → detik ──
def parse_duration_to_seconds(val):
    """Konversi string durasi '0:04:55' atau '4:55' ke detik (int)."""
    if pd.isna(val) or val == '':
        return np.nan
    val = str(val).strip()
    parts = val.split(':')
    try:
        parts = [int(p) for p in parts]
        if len(parts) == 3:   # H:MM:SS
            return parts[0]*3600 + parts[1]*60 + parts[2]
        elif len(parts) == 2: # MM:SS
            return parts[0]*60 + parts[1]
        else:
            return int(parts[0])
    except:
        return np.nan

# Rata-rata durasi tonton → detik
if 'Rata-rata durasi tonton' in df.columns:
    df['avg_watch_seconds'] = df['Rata-rata durasi tonton'].apply(parse_duration_to_seconds)
    print("avg_watch_seconds dibuat")

# Durasi video (sudah dalam detik di dataset ini)
df['durasi_detik'] = pd.to_numeric(df['Durasi'], errors='coerce')

# Waktu publikasi → datetime
if 'Waktu publikasi video' in df.columns:
    df['published_at'] = pd.to_datetime(df['Waktu publikasi video'], errors='coerce')
    print("published_at dibuat")

# Kolom numerik — pastikan float
numeric_cols = [
    'Penayangan', 'Tayangan', 'Suka', 'Tidak suka', 'Komentar ditambahkan',
    'Pembagian', 'Subscriber yang diperoleh', 'Subscriber yang hilang',
    'Poin hype', 'Estimasi pendapatan (IDR)', 'RPM (IDR)', 'CPM (IDR)',
    'Rasio klik-tayang dari tayangan (%)', 'Persentase penayangan rata-rata (%)',
    'Penonton unik', 'Waktu tonton (jam)'
]
existing_numeric = [c for c in numeric_cols if c in df.columns]
df[existing_numeric] = df[existing_numeric].apply(pd.to_numeric, errors='coerce')
print(f" {len(existing_numeric)} kolom numerik dikonversi")

### 2.4 Hapus baris yang tidak memiliki data inti

In [ ]:
# Kolom WAJIB ada nilainya
required_cols = ['Penayangan', 'durasi_detik', 'Tayangan']
existing_required = [c for c in required_cols if c in df.columns]

before = len(df)
df = df.dropna(subset=existing_required)
after = len(df)

print(f"  Hapus {before - after} baris tanpa data inti")
print(f"   Sisa: {after} video")

### 2.5 Hapus data outlier ekstrem (Penayangan = 0)

In [ ]:
before = len(df)
df = df[df['Penayangan'] > 0]
print(f"  Hapus {before - len(df)} video dengan 0 penayangan")
print(f"   Sisa: {len(df)} video")

### 2.6 Ringkasan data bersih

In [ ]:
print("=" * 50)
print(f" Data Bersih: {len(df)} video, {df.shape[1]} kolom")
print("=" * 50)

# Null summary setelah cleaning
null_remaining = df.isnull().mean() * 100
null_remaining = null_remaining[null_remaining > 0].sort_values(ascending=False)
print("\nNull yang masih tersisa (kolom yang ada null-nya):")
print(null_remaining.head(15))

---
## 3. Feature Engineering

### 3.1 Engagement Ratios

In [ ]:
views = df['Penayangan'].replace(0, np.nan)  # hindari division by zero

# Like rate, comment rate, share rate
if 'Suka' in df.columns:
    df['like_rate'] = df['Suka'] / views

if 'Komentar ditambahkan' in df.columns:
    df['comment_rate'] = df['Komentar ditambahkan'] / views

if 'Pembagian' in df.columns:
    df['share_rate'] = df['Pembagian'] / views

if 'Tidak suka' in df.columns and 'Suka' in df.columns:
    total_reactions = df['Suka'] + df['Tidak suka']
    df['dislike_ratio'] = df['Tidak suka'] / total_reactions.replace(0, np.nan)

# Overall engagement rate (like + comment + share per view)
engagement_cols = [c for c in ['Suka', 'Komentar ditambahkan', 'Pembagian'] if c in df.columns]
df['engagement_rate'] = df[engagement_cols].sum(axis=1) / views

print("   Engagement ratios dibuat:")
print("   like_rate, comment_rate, share_rate, dislike_ratio, engagement_rate")

### 3.2 Watch Time & Retention Metrics

In [ ]:
# Watch Time Ratio: rata-rata durasi tonton / durasi video
# → 1.0 = ditonton full, 0.3 = hanya 30% ditonton
df['watch_time_ratio'] = df['avg_watch_seconds'] / df['durasi_detik'].replace(0, np.nan)
df['watch_time_ratio'] = df['watch_time_ratio'].clip(0, 1)  # cap di 1.0

# Revenue per View (proxy kualitas penonton)
if 'Estimasi pendapatan (IDR)' in df.columns:
    df['revenue_per_view'] = df['Estimasi pendapatan (IDR)'] / views

# Subscriber Net (subscriber bersih per video)
sub_gain = 'Subscriber yang diperoleh'
sub_lost = 'Subscriber yang hilang'
if sub_gain in df.columns and sub_lost in df.columns:
    df['subscriber_net'] = df[sub_gain] - df[sub_lost].fillna(0)
    df['subscriber_net_per_view'] = df['subscriber_net'] / views

print(" Watch time & retention metrics dibuat:")
print("   watch_time_ratio, revenue_per_view, subscriber_net, subscriber_net_per_view")

### 3.3 Impression & CTR Metrics

In [ ]:
# CTR (sudah ada di data): Rasio klik-tayang dari tayangan (%)
df['ctr'] = df['Rasio klik-tayang dari tayangan (%)'] / 100  # ubah ke proporsi 0–1

# Impression-to-View Rate: seberapa efektif tayangan → jadi view
if 'Tayangan' in df.columns:
    df['impression_to_view_rate'] = df['Penayangan'] / df['Tayangan'].replace(0, np.nan)

print("Impression & CTR metrics dibuat:")
print("ctr (0-1), impression_to_view_rate")

### 3.4 Fitur Waktu Publikasi

In [ ]:
if 'published_at' in df.columns:
    df['upload_hour']    = df['published_at'].dt.hour
    df['upload_day']     = df['published_at'].dt.dayofweek  # 0=Senin, 6=Minggu
    df['upload_day_name']= df['published_at'].dt.day_name()
    df['upload_month']   = df['published_at'].dt.month
    df['upload_year']    = df['published_at'].dt.year

    # Umur video dalam hari (dari tanggal upload sampai sekarang)
    today = pd.Timestamp.today().normalize()
    df['video_age_days'] = (today - df['published_at'].dt.normalize()).dt.days

    print(" Fitur waktu publikasi dibuat:")
    print("   upload_hour, upload_day (0=Senin), upload_day_name, upload_month, upload_year, video_age_days")

### 3.5 Duration Bucket (Kategori Panjang Video)

In [ ]:
def categorize_duration(seconds):
    """Kategorikan durasi video ke bucket yang bermakna untuk algoritma YouTube."""
    if pd.isna(seconds):
        return 'unknown'
    minutes = seconds / 60
    if minutes < 1:
        return 'shorts (<1 mnt)'
    elif minutes < 3:
        return 'pendek (1-3 mnt)'
    elif minutes < 7:
        return 'medium (3-7 mnt)'
    elif minutes < 15:
        return 'panjang (7-15 mnt)'
    else:
        return 'sangat panjang (>15 mnt)'

df['duration_bucket'] = df['durasi_detik'].apply(categorize_duration)

print("Duration bucket dibuat:")
print(df['duration_bucket'].value_counts())

### 3.6 NLP Fitur dari Judul Video

In [ ]:
# Kata-kata sensasional/clickbait yang umum di konten berita/geo-politik Indonesia
SENSATIONAL_WORDS = [
    'GEMPAR', 'HEBOH', 'NGAMUK', 'PANIK', 'MERINDING', 'GEGER', 'VIRAL',
    'MENGEJUTKAN', 'MENGEJUTKAN', 'BONGKAR', 'HANCUR', 'TERDIAM', 'KAGET',
    'MALU', 'TAKUT', 'MARAH', 'MENGGUNCANG', 'TERHINA', 'BOIKOT', 'PERANG',
    'GILA', 'LUAR BIASA', 'BERANI', 'NEKAT', 'HAJAR', 'BANTAI'
]

# Topik/cluster konten
TOPIC_KEYWORDS = {
    'israel_palestina': ['ISRAEL', 'PALESTINA', 'GAZA', 'HAMAS', 'IDF'],
    'malaysia':         ['MALAYSIA', 'MELAYU', 'JIRAN'],
    'militer_tni':      ['TNI', 'MILITER', 'ALUTSISTA', 'ANGKATAN', 'KAPAL PERANG', 'TANK', 'RUDAL'],
    'ekonomi_mineral':  ['MINERAL', 'NIKEL', 'BATU BARA', 'EKONOMI', 'INVESTASI', 'EKSPOR'],
    'diplomasi':        ['PRABOWO', 'DIPLOMATIK', 'PBB', 'SIDANG', 'HUBUNGAN', 'KTT'],
    'amerika_barat':    ['AMERIKA', 'AS', 'USA', 'BARAT', 'NATO', 'CIA'],
    'cina':             ['CINA', 'CHINA', 'TIONGKOK', 'XI JINPING'],
    'rusia':            ['RUSIA', 'RUSSIA', 'PUTIN'],
}

def extract_title_features(title):
    """Extract NLP features dari judul video."""
    if pd.isna(title):
        return {}

    title_upper = str(title).upper()
    title_str   = str(title)

    features = {}

    # 1. Panjang judul
    features['title_length']  = len(title_str)
    features['title_words']   = len(title_str.split())

    # 2. Jumlah kata sensasional
    features['sensational_count'] = sum(
        1 for w in SENSATIONAL_WORDS if w in title_upper
    )

    # 3. Apakah ada simbol/emoji clickbait
    features['has_symbol'] = int(bool(re.search(r'[‼⁉✅🚨🔥❗❓💥🇮🇩⚡]', title_str)))

    # 4. Proporsi huruf kapital (dari karakter huruf saja)
    letters = [c for c in title_str if c.isalpha()]
    if letters:
        features['caps_ratio'] = sum(1 for c in letters if c.isupper()) / len(letters)
    else:
        features['caps_ratio'] = 0.0

    # 5. Topic cluster (multi-label, pakai yang paling banyak keyword-nya)
    topic_scores = {}
    for topic, keywords in TOPIC_KEYWORDS.items():
        topic_scores[topic] = sum(1 for kw in keywords if kw in title_upper)

    best_topic = max(topic_scores, key=topic_scores.get)
    features['topic_cluster'] = best_topic if topic_scores[best_topic] > 0 else 'lainnya'
    features['topic_score']   = topic_scores[best_topic]

    # 6. Jumlah entitas negara/tokoh yang disebut
    all_entities = [kw for kws in TOPIC_KEYWORDS.values() for kw in kws]
    features['entity_count'] = sum(1 for e in all_entities if e in title_upper)

    return features

# Terapkan ke semua judul
title_features = df['Judul video'].apply(extract_title_features)
title_df = pd.DataFrame(title_features.tolist(), index=df.index)
df = pd.concat([df, title_df], axis=1)

print(" NLP fitur judul dibuat:")
print(" title_length, title_words, sensational_count, has_symbol,")
print(" caps_ratio, topic_cluster, topic_score, entity_count")
print(f"\nDistribusi topic_cluster:")
print(df['topic_cluster'].value_counts())

---
## 4. Performance Classification

Klasifikasi performa video berdasarkan **Views × Watch Time Ratio**.

Logika: Video bagus = banyak yang nonton DAN menontonnya lama.
- `1000 views × 2 mnt` > `500 views × 1 mnt`
- Score = `Penayangan × (avg_watch_seconds / 60)`

In [ ]:

# PERFORMANCE SCORE: Views × Rata-rata waktu tonton (menit)

df['avg_watch_minutes'] = df['avg_watch_seconds'] / 60

# Score utama: Views × Avg Watch Minutes
# Contoh: 1000 views × 2 mnt = 2000 | 500 views × 1 mnt = 500
df['perf_score'] = df['Penayangan'] * df['avg_watch_minutes']

print("Distribusi perf_score:")
print(df['perf_score'].describe())

In [ ]:
# PERFORMANCE CLASS berdasarkan persentil channel sendiri
# Lebih adil daripada threshold absolut karena
# channel besar dan kecil punya skala berbeda

p20 = df['perf_score'].quantile(0.20)  # 20% terbawah = Rendah
p50 = df['perf_score'].quantile(0.50)  # Median
p80 = df['perf_score'].quantile(0.80)  # 20% teratas = Bagus
p95 = df['perf_score'].quantile(0.95)  # 5% teratas = Viral

print(f"Persentil perf_score channel kamu:")
print(f"  P20 (batas Rendah):  {p20:,.0f}")
print(f"  P50 (Median):        {p50:,.0f}")
print(f"  P80 (batas Bagus):   {p80:,.0f}")
print(f"  P95 (batas Viral):   {p95:,.0f}")

def assign_performance_class(score):
    """
    Klasifikasi performa video berdasarkan Views × Watch Time.

    Tier:
       rendah   — P0–P20   (underperforming)
       sedang   — P20–P50  (di bawah median)
       bagus    — P50–P80  (di atas median)
       sangat_bagus — P80–P95  (top performers)
       viral    — P95+     (viral / outlier positif)
    """
    if pd.isna(score):
        return 'unknown'
    elif score < p20:
        return 'rendah'
    elif score < p50:
        return 'sedang'
    elif score < p80:
        return 'bagus'
    elif score < p95:
        return 'sangat_bagus'
    else:
        return 'viral'

df['performance_class'] = df['perf_score'].apply(assign_performance_class)

print("\n performance_class dibuat!")
print("\nDistribusi kelas:")
class_order = ['rendah', 'sedang', 'bagus', 'sangat_bagus', 'viral']
print(df['performance_class'].value_counts().reindex(class_order))

In [ ]:
# Visualisasi distribusi kelas performa
print("\n Ringkasan per kelas performa:")
print("-" * 80)

summary_cols = ['Penayangan', 'avg_watch_minutes', 'perf_score', 'ctr', 'like_rate', 'watch_time_ratio']
existing_summary = [c for c in summary_cols if c in df.columns]

summary = df.groupby('performance_class')[existing_summary].median().reindex(class_order)
print(summary.to_string())

### 4.1 Contoh Threshold Absolut (Opsional)

Kalau mau pakai threshold tetap (bukan persentil), uncomment kode di bawah ini dan sesuaikan angkanya:

In [ ]:
# # Threshold absolut — sesuaikan dengan karakteristik channel kamu
# THRESHOLDS = {
#     'viral':       {'min_views': 500_000, 'min_watch_min': 3.0},
#     'sangat_bagus':{'min_views': 200_000, 'min_watch_min': 2.5},
#     'bagus':       {'min_views': 100_000, 'min_watch_min': 2.0},
#     'sedang':      {'min_views': 50_000,  'min_watch_min': 1.5},
#     'rendah':      {'min_views': 0,       'min_watch_min': 0.0},
# }

# def classify_absolute(row):
#     v = row.get('Penayangan', 0)
#     w = row.get('avg_watch_minutes', 0)
#     if v >= 500_000 and w >= 3.0: return 'viral'
#     elif v >= 200_000 and w >= 2.5: return 'sangat_bagus'
#     elif v >= 100_000 and w >= 2.0: return 'bagus'
#     elif v >= 50_000 and w >= 1.5: return 'sedang'
#     else: return 'rendah'

# df['performance_class_absolute'] = df.apply(classify_absolute, axis=1)
print("(Threshold absolut — uncomment untuk dipakai)")

---
## 5. Ringkasan Fitur Final

In [ ]:
# Fitur yang akan dipakai untuk ML
ML_FEATURES = [
    # Identitas
    'Konten',           # video ID
    'Judul video',
    'published_at',

    # Raw Metrics (sudah ada di data)
    'Penayangan',
    'Tayangan',
    'durasi_detik',
    'avg_watch_seconds',
    'Poin hype',
    'RPM (IDR)',
    'CPM (IDR)',
    'Persentase penayangan rata-rata (%)',

    # Fitur Engineered
    'like_rate',
    'comment_rate',
    'share_rate',
    'dislike_ratio',
    'engagement_rate',
    'watch_time_ratio',
    'revenue_per_view',
    'subscriber_net',
    'subscriber_net_per_view',
    'ctr',
    'impression_to_view_rate',
    'upload_hour',
    'upload_day',
    'upload_day_name',
    'upload_month',
    'upload_year',
    'video_age_days',
    'duration_bucket',

    # NLP Features
    'title_length',
    'title_words',
    'sensational_count',
    'has_symbol',
    'caps_ratio',
    'topic_cluster',
    'topic_score',
    'entity_count',

    # Target Variable
    'perf_score',
    'performance_class',
]

existing_ml_features = [f for f in ML_FEATURES if f in df.columns]
df_ml = df[existing_ml_features].copy()

print(f" Dataset ML siap: {df_ml.shape[0]} video × {df_ml.shape[1]} fitur")
print("\nFitur tersedia:")
for f in existing_ml_features:
    null_pct = df_ml[f].isnull().mean() * 100
    dtype = df_ml[f].dtype
    print(f"  {f:<40} {str(dtype):<12} null: {null_pct:.1f}%")

---
## 6. Simpan Hasil

In [ ]:
# Simpan dataset bersih + fitur lengkap
OUTPUT_FULL = 'data_cleaned_full.csv'
OUTPUT_ML   = 'data_ml_features.csv'

df.to_csv(OUTPUT_FULL, index=False)
df_ml.to_csv(OUTPUT_ML, index=False)

print(f" Disimpan:")
print(f"   {OUTPUT_FULL}  → {df.shape[0]} baris, {df.shape[1]} kolom (semua kolom)")
print(f"   {OUTPUT_ML}    → {df_ml.shape[0]} baris, {df_ml.shape[1]} kolom (fitur ML saja)")

In [ ]:
# Preview 5 baris terakhir
print("\n🔍 Sample data ML:")
preview_cols = ['Judul video', 'Penayangan', 'avg_watch_minutes',
                'perf_score', 'performance_class', 'topic_cluster', 'ctr']
existing_preview = [c for c in preview_cols if c in df_ml.columns]
df_ml[existing_preview].head(10)

---
## 7. Catatan & Next Steps

### Yang sudah selesai:
-  Hapus baris Total & baris kosong
-  Drop kolom >80% null
-  Konversi tipe data (waktu tonton, durasi, datetime)
-  Engineering: engagement rates, watch time ratio, CTR, subscriber net
-  Fitur waktu publikasi (jam, hari, bulan, umur video)
-  Duration bucket
-  NLP fitur dari judul (topic cluster, sensational count, dll)
-  **Performance class** berbasis Views × Watch Time (5 kelas)

### Next Steps untuk ML:
1. **Model 1 — Classifier**: XGBoost/LightGBM untuk prediksi `performance_class`
2. **Model 2 — Regressor**: Prediksi total views
3. **Model 3 — SHAP Analysis**: Kenapa video underperform?
4. **Data tambahan**: Export time series harian per video dari YouTube Studio untuk rolling average & velocity decay
